# Multi-Model Exception Prediction Training

## 1. Import Libraries

In [2]:

import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier


## 2. Load Training Dataset

In [3]:

df = pd.read_csv("structured_training_dataset.csv")

print(df.shape)

df.head()


(1000, 19)


,postcode,phone_available,email_available,preferred_contact,past_no_contact_count,past_no_access_count,total_contact_attempts,field_visit_attempts,cancellation_before_visit,appointment_slot,days_between_booking_and_visit,reschedule_count,property_type,shared_access,meter_location_current,meter_location_target,no_access_exception,dig_exception,survey_exception
0,PC11,False,True,Call,4,0,1,1,True,Morning,17,0,House,False,Internal,Internal,0,0,0
1,PC05,False,True,Call,1,2,4,2,True,Afternoon,12,1,Commercial,True,Internal,External,0,1,1
2,PC09,True,True,Mail,2,1,3,0,False,Evening,12,0,House,False,Internal,Internal,0,0,1
3,PC11,False,True,SMS,4,2,1,3,False,Afternoon,9,1,House,True,Internal,External,0,0,0
4,PC15,False,False,Email,4,1,2,1,True,Evening,16,2,Commercial,False,Internal,External,1,0,1


## 3. Define Feature Groups

In [4]:

NO_ACCESS_FEATURES = [
    "postcode",
    "phone_available",
    "email_available",
    "preferred_contact",
    "past_no_contact_count",
    "past_no_access_count",
    "total_contact_attempts",
    "field_visit_attempts",
    "cancellation_before_visit",
    "appointment_slot",
    "days_between_booking_and_visit",
    "reschedule_count",
    "property_type",
    "shared_access",
    "meter_location_current",
    "meter_location_target"
]

DIG_FEATURES = [
    "postcode",
    "property_type",
    "shared_access",
    "field_visit_attempts",
    "days_between_booking_and_visit",
    "meter_location_current",
    "meter_location_target"
]

SURVEY_FEATURES = [
    "postcode",
    "property_type",
    "shared_access",
    "field_visit_attempts",
    "past_no_access_count",
    "days_between_booking_and_visit",
    "reschedule_count",
    "meter_location_current",
    "meter_location_target"
]


## 4. Model Training Function

In [5]:

def train_model(features, target):

    X = df[features]
    y = df[target]

    categorical_cols = X.select_dtypes(include="object").columns.tolist()

    numerical_cols = [
        col for col in X.columns
        if col not in categorical_cols
    ]

    preprocessor = ColumnTransformer([
        (
            "cat",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="most_frequent")
                ),
                (
                    "encoder",
                    OneHotEncoder(handle_unknown="ignore")
                )
            ]),
            categorical_cols
        ),

        (
            "num",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="median")
                )
            ]),
            numerical_cols
        )
    ])

    model = Pipeline([
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=200,
                random_state=42
            )
        )
    ])

    model.fit(X, y)

    return model


## 5. Train Models

In [6]:

no_access_model = train_model(
    NO_ACCESS_FEATURES,
    "no_access_exception"
)

dig_model = train_model(
    DIG_FEATURES,
    "dig_exception"
)

survey_model = train_model(
    SURVEY_FEATURES,
    "survey_exception"
)

print("Training Complete")


Training Complete


## 6. Export Pickle Files

In [7]:

joblib.dump(
    no_access_model,
    "no_access_model.pkl"
)

joblib.dump(
    dig_model,
    "dig_model.pkl"
)

joblib.dump(
    survey_model,
    "survey_model.pkl"
)

print("All models exported successfully.")


All models exported successfully.
